# Backpropagation from Scratch

Companion notebook for Section 6 of the MLP lecture notes.

Objectives:

- reuse the 2-2-1 network from the forward-pass notebook;
- compute output and hidden error signals manually;
- compute gradients for all weights and biases;
- apply one gradient-descent update and verify that the loss decreases;
- validate manual gradients with finite differences and, if available, PyTorch autograd.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (7, 4)
plt.rcParams['axes.grid'] = True
np.set_printoptions(precision=6, suppress=True)


## 1. Same network as the lecture example

We use sigmoid activations and the loss `L = 0.5 * (y_hat - y)^2`.


In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def d_sigmoid_from_activation(a):
    return a * (1 - a)

def loss_half_mse(y_hat, y):
    return 0.5 * (y_hat - y) ** 2

x = np.array([0.05, 0.10])
y = 0.01

W1 = np.array([[0.15, 0.20],
               [0.25, 0.30]], dtype=float)
b1 = np.array([0.35, 0.35], dtype=float)
W2 = np.array([[0.40, 0.45]], dtype=float)
b2 = np.array([0.60], dtype=float)


## 2. Forward pass with cached values

Backpropagation needs the pre-activations and activations computed during the forward pass.


In [ ]:
def forward(x, W1, b1, W2, b2):
    z1 = W1 @ x + b1
    a1 = sigmoid(z1)
    z2 = W2 @ a1 + b2
    a2 = sigmoid(z2)
    loss = loss_half_mse(a2[0], y)
    return {'x': x, 'z1': z1, 'a1': a1, 'z2': z2, 'a2': a2, 'loss': loss}

cache = forward(x, W1, b1, W2, b2)
for key, value in cache.items():
    print(f'{key}: {value}')


## 3. Output-layer error signal

The output error signal is

`delta2 = dL/dz2 = dL/da2 * da2/dz2`.

For MSE and sigmoid output: `delta2 = (a2 - y) * a2 * (1 - a2)`.


In [ ]:
a2 = cache['a2']
a1 = cache['a1']

delta2 = (a2 - y) * d_sigmoid_from_activation(a2)
print('delta2:', delta2)


## 4. Gradients for the output layer

Each weight gradient is destination error signal times incoming activation.


In [ ]:
dW2 = delta2.reshape(1, 1) @ a1.reshape(1, -1)
db2 = delta2.copy()

print('dW2:', dW2)
print('db2:', db2)


## 5. Hidden-layer error signals

The hidden layer receives the output error propagated backward through `W2`, then multiplied by the local sigmoid derivative.


In [ ]:
delta1 = (W2.T @ delta2).ravel() * d_sigmoid_from_activation(a1)
print('delta1:', delta1)


## 6. Gradients for the hidden layer

For the first layer, each hidden neuron's error signal multiplies each original input.


In [ ]:
dW1 = delta1.reshape(-1, 1) @ x.reshape(1, -1)
db1 = delta1.copy()

print('dW1:\n', dW1)
print('db1:', db1)


## 7. One gradient-descent update

After one update with learning rate `eta = 0.5`, the loss should decrease.


In [ ]:
eta = 0.5
before = cache['loss']

W1_new = W1 - eta * dW1
b1_new = b1 - eta * db1
W2_new = W2 - eta * dW2
b2_new = b2 - eta * db2

after_cache = forward(x, W1_new, b1_new, W2_new, b2_new)
after = after_cache['loss']

print('Loss before update:', before)
print('Loss after update:', after)
print('Loss decreased?', after < before)


## 8. Compact backward function

This wraps the same calculation in a reusable function.


In [ ]:
def backward(cache, W2, y):
    x = cache['x']
    a1 = cache['a1']
    a2 = cache['a2']

    delta2 = (a2 - y) * d_sigmoid_from_activation(a2)
    dW2 = delta2.reshape(1, 1) @ a1.reshape(1, -1)
    db2 = delta2.copy()

    delta1 = (W2.T @ delta2).ravel() * d_sigmoid_from_activation(a1)
    dW1 = delta1.reshape(-1, 1) @ x.reshape(1, -1)
    db1 = delta1.copy()

    return {'dW1': dW1, 'db1': db1, 'dW2': dW2, 'db2': db2, 'delta1': delta1, 'delta2': delta2}

grads = backward(cache, W2, y)
for key, value in grads.items():
    print(f'{key}:\n{value}')


## 9. Finite-difference gradient check

A numerical gradient check perturbs each parameter slightly and estimates the slope of the loss. It is slower than backpropagation, but useful for debugging small implementations.


In [ ]:
def scalar_loss(W1, b1, W2, b2):
    return forward(x, W1, b1, W2, b2)['loss']

def finite_difference(param_name, index, eps=1e-6):
    params_plus = {'W1': W1.copy(), 'b1': b1.copy(), 'W2': W2.copy(), 'b2': b2.copy()}
    params_minus = {'W1': W1.copy(), 'b1': b1.copy(), 'W2': W2.copy(), 'b2': b2.copy()}
    params_plus[param_name][index] += eps
    params_minus[param_name][index] -= eps
    loss_plus = scalar_loss(**params_plus)
    loss_minus = scalar_loss(**params_minus)
    return (loss_plus - loss_minus) / (2 * eps)

checks = []
for name, manual in [('W1', dW1), ('b1', db1), ('W2', dW2), ('b2', db2)]:
    for index in np.ndindex(manual.shape):
        numeric = finite_difference(name, index)
        checks.append({
            'parameter': name,
            'index': index,
            'manual_gradient': manual[index],
            'numeric_gradient': numeric,
            'absolute_error': abs(manual[index] - numeric),
        })

check_df = pd.DataFrame(checks)
check_df


In [ ]:
print('Maximum absolute gradient-check error:', check_df['absolute_error'].max())


## 10. Optional PyTorch autograd check

If PyTorch is installed, the next cell verifies the same gradients using automatic differentiation. If it is not installed, the cell reports how to enable the check.


In [ ]:
try:
    import torch
    TORCH_AVAILABLE = True
except ImportError:
    TORCH_AVAILABLE = False

print('PyTorch available:', TORCH_AVAILABLE)
if not TORCH_AVAILABLE:
    print('Install PyTorch to run the autograd check, then re-run this notebook.')


In [ ]:
if TORCH_AVAILABLE:
    dtype = torch.float64
    x_t = torch.tensor(x, dtype=dtype)
    y_t = torch.tensor([y], dtype=dtype)
    W1_t = torch.tensor(W1, dtype=dtype, requires_grad=True)
    b1_t = torch.tensor(b1, dtype=dtype, requires_grad=True)
    W2_t = torch.tensor(W2, dtype=dtype, requires_grad=True)
    b2_t = torch.tensor(b2, dtype=dtype, requires_grad=True)

    a1_t = torch.sigmoid(W1_t @ x_t + b1_t)
    a2_t = torch.sigmoid(W2_t @ a1_t + b2_t)
    loss_t = 0.5 * (a2_t[0] - y_t[0]) ** 2
    loss_t.backward()

    print('dW1 matches:', np.allclose(W1_t.grad.detach().numpy(), dW1))
    print('db1 matches:', np.allclose(b1_t.grad.detach().numpy(), db1))
    print('dW2 matches:', np.allclose(W2_t.grad.detach().numpy(), dW2))
    print('db2 matches:', np.allclose(b2_t.grad.detach().numpy(), db2))
else:
    print('Skipped because PyTorch is not installed.')


## Exercises

1. Change `eta` to 5.0. Does the first update still reduce the loss?
2. Replace the output sigmoid with a linear output. How do the error signal formulas change?
3. Add one more hidden neuron and derive the new shapes of `dW1` and `dW2`.


## Takeaways

- Backpropagation computes error signals from output to input.
- Weight gradients factor into incoming activation times destination error signal.
- The optimizer is separate from backpropagation: it uses gradients to update parameters.
- Finite differences are useful for checking a small manual gradient implementation.
